# truthy_falsy
Truthy and falsy behavior drives a huge amount of Python control flow. Many production bugs happen when engineers treat falsey values as “missing” even though `0`, `[]`, `""`, and `None` often mean very different things.

## 1. Problem First
Why does this matter in real systems?
- A timeout of `0` may be valid but gets replaced by a default.
- An empty list may mean “no results,” not “bad input.”
- A missing config value and an explicit false value should not be treated the same.

In [1]:
timeout = 0
if timeout:
    print("custom timeout")
else:
    print("fallback timeout")

fallback timeout


## 2. Minimal Concept
Python treats some values as false in condition checks.
- Falsey examples: `False`, `None`, `0`, `0.0`, `""`, `[]`, `{}`, `set()`
- Most other objects are truthy.
- `if value:` uses truthiness, not strict boolean type checking.

## 3. Mental Model
How Python actually behaves
When Python evaluates a condition, it does not ask only whether the value is literally `True` or `False`. It asks whether the object should behave as true or false in a boolean context. That means truthiness is about control-flow semantics, not just type.

In [2]:
values = [False, None, 0, "", [], [1], "hello"]
for value in values:
    print(repr(value), bool(value))

False False
None False
0 False
'' False
[] False
[1] True
'hello' True


## 4. Internal Mechanics
Python checks truthiness by calling `bool(value)` behavior under the hood. For built-in containers, emptiness usually means false. For custom objects, truthiness can be controlled with `__bool__()` or `__len__()`.

In [3]:
import dis

class Batch:
    def __init__(self, records):
        self.records = records
    def __len__(self):
        return len(self.records)

def should_process(batch):
    if batch:
        return "process"
    return "skip"

dis.dis(should_process)
print(should_process(Batch([])))
print(should_process(Batch([1, 2])))

  9           0 RESUME                   0

 10           2 LOAD_FAST                0 (batch)
              4 POP_JUMP_IF_FALSE        1 (to 8)

 11           6 RETURN_CONST             1 ('process')

 12     >>    8 RETURN_CONST             2 ('skip')
skip
process


## 5. Edge Cases
Where it breaks:
- `0` may be a valid configured value, not a missing one.
- `[]` can mean “empty result” rather than “bad request.”
- `or` defaults can silently overwrite legitimate falsey values.
- Custom objects may define surprising truthiness.

In [4]:
configured_port = 0
port = configured_port or 8080
print(port)

records = []
if records:
    print("has records")
else:
    print("empty result set")

8080
empty result set


## 6. Performance Thinking
- Truthiness checks are usually O(1).
- Container truthiness may depend on length, which is still cheap for built-ins.
- The main cost is correctness: falsey confusion causes control-flow bugs, not performance issues.
- Clear explicit checks often save more debugging time than micro-optimizations.

## 7. Real Use Case
A config loader must distinguish between missing values and intentionally provided falsey values.

In [5]:
def resolve_retries(raw_retries):
    if raw_retries is None:
        return 3
    return raw_retries

print(resolve_retries(None))
print(resolve_retries(0))

3
0


## 8. Anti-Pattern
What beginners do wrong:
- Use `if value` when the real check is `is None`.
- Treat all empty values as invalid input.
- Use `value or default` in config code without thinking about zero and empty string.

In [6]:
def bad_timeout(configured_timeout):
    return configured_timeout or 30

print(bad_timeout(0))

def good_timeout(configured_timeout):
    if configured_timeout is None:
        return 30
    return configured_timeout

print(good_timeout(0))

30
0


## 9. Interview Signals
Questions FAANG asks:
- What values are falsey in Python?
- When should you use `if value` versus `if value is None`?
- Why can `value or default` be dangerous?
- How can custom classes control truthiness?

## 10. Exercise (Non-trivial)
You are designing a config parser for a CLI tool. Inputs may be `None`, `0`, `False`, `""`, or empty lists. Decide which values mean “missing,” which mean “explicitly provided,” and write the branching so no legitimate value gets overwritten by defaults.

In [7]:
def resolve_config_value(value, default):
    return value or default

# Task:
# 1. Preserve valid falsey values.
# 2. Only treat missing values as default-worthy.
# 3. Explain why truthiness alone is not enough here.